In [178]:
import requests
import json
import os
from dotenv import load_dotenv
import pandas as pd
from pydantic_ai import Agent, RunContext
from pydantic_ai.models.openrouter import OpenRouterModel, OpenRouterModelSettings
from pydantic_ai.messages import ModelResponse, ToolCallPart, ToolReturnPart, ModelRequest
from pydantic_ai.providers.openrouter import OpenRouterProvider
from pydantic_ai.capabilities import MCP
from pydantic import BaseModel
import httpx
import ast
load_dotenv()

True

# Helper Functions fürs Logging

In [179]:
def log_mcp_usage(response):

    mcp_calls = []

    # 1) Alle ToolCallParts aus den ModelResponses sammeln
    for msg in response.all_messages():
        if isinstance(msg, ModelResponse):
            for part in msg.parts:
                if isinstance(part, ToolCallPart):
                    mcp_calls.append({
                        "tool_call_id": part.tool_call_id,
                        "tool_name": part.tool_name,
                        "args": part.args_as_dict(),
                        "result": None,
                        "outcome": None,
                    })
    returns_by_type = {}
    for call in mcp_calls:
        returns_by_type.setdefault(call["tool_name"], []).append(call)

    mcp_call_counts = {tool_name: len(calls) for tool_name, calls in returns_by_type.items()}
    return mcp_call_counts

In [235]:
input_price_pMt = 0.09
output_price_pMt = 0.18

In [236]:
# Logging der Agent-Aktivitäten
def monitor_tokens(response, input_price_pMt=input_price_pMt, output_price_pMt=output_price_pMt):
    
    # Message History auslesen
    if hasattr(response, 'messages_history'):
        messages = response.messages_history
    else:
        messages = response.all_messages() if hasattr(response, 'all_messages') else []
    
    input_tokens = response.usage.input_tokens if response.usage else 'N/A'
    output_tokens = response.usage.output_tokens if response.usage else 'N/A'
    costs = (input_tokens * input_price_pMt + output_tokens * output_price_pMt)/1000000 if response.usage else 'N/A'
    return {
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "costs": costs,
        "mcp_usage": log_mcp_usage(response)
    }
    

# Agent inkl MCP Setup

In [237]:
poliscope_mcp=f'https://api.poliscope.de/v2/mcp/poliscope?access_token={os.getenv("POLISCOPE_API_KEY")}'

In [238]:
with open('./system_prompt.txt', 'r') as file:
    system_prompt = file.read().replace('\n', '')

In [239]:
# OpenRouter Konfiguration für Gemini Flash
OPEN_ROUTER_KEY = os.getenv("OPEN_ROUTER_KEY")
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

model = OpenRouterModel(
    "deepseek/deepseek-v4-flash",
    provider=OpenRouterProvider(api_key=OPEN_ROUTER_KEY),
)
settings = OpenRouterModelSettings(
    openrouter_reasoning={
        'effort': 'high',
    },
    openrouter_usage={
        'include': True,
    }
)
# Agent mit Gemini Flash über OpenRouter
agent = Agent(
    model,
    model_settings=settings,
    system_prompt=system_prompt,
   #capabilities=[MCP(poliscope_mcp)]
)

# Analyse

In [247]:
data = pd.read_csv("./data/raw/big_cities_heat_planning.csv")

In [248]:
data

,groupKey,groupType,groupCreationDate,date,score,hits,totalHits,context
0,proposal:5a25a544-c1ea-4e44-95d4-0d7c4b2a5264,proposal,2025-12-03T13:00:00.000Z,2025-12-03T14:00:00,2.346608,[{'id': 'proposalDescription:5a25a544-c1ea-4e4...,3,"{'date': '2025-12-03T14:00:00', 'entityName': ..."
1,proposal:e467ab79-b7e4-4b19-8384-fa1840d914eb,proposal,2025-11-13T17:00:00.000Z,2025-11-13T18:00:00,2.346608,[{'id': 'proposalDescription:e467ab79-b7e4-4b1...,3,"{'date': '2025-11-13T18:00:00', 'entityName': ..."
2,proposal:00fb5711-22de-449a-a71e-9a1c0a9cb2e8,proposal,2025-12-03T13:00:00.000Z,2025-12-03T14:00:00,2.229086,[{'id': 'proposalDescription:00fb5711-22de-449...,3,"{'date': '2025-12-03T14:00:00', 'entityName': ..."
3,meeting:7b3225df-341d-437d-9307-7a243b03d86a,meeting,2025-04-01T14:00:00.000Z,2025-04-01T16:00:00,2.102905,[{'id': 'agendaItemDescription:a9a0d66b-1a2f-4...,3,"{'date': '2025-04-01T16:00:00', 'entityName': ..."
4,proposal:58bd4b70-d34f-45ac-9d4b-f68de914646f,proposal,2026-05-29T10:18:30.428Z,2026-05-28T16:00:00,2.031953,[{'id': 'proposalDescription:58bd4b70-d34f-45a...,3,"{'date': '2026-05-28T16:00:00', 'entityName': ..."
...,...,...,...,...,...,...,...,...
184,proposal:558ab72a-badb-4452-a05c-227e6c97220c,proposal,2024-02-07T16:00:00.000Z,2024-02-07T17:00:00,0.309111,[{'id': 'proposalDescription:558ab72a-badb-445...,1,"{'date': '2024-02-07T17:00:00', 'entityName': ..."
185,meeting:ad287353-f301-4b34-b859-96e035dc8ae1,meeting,2025-07-08T16:30:00.000Z,2025-07-08T18:30:00,0.304059,[{'id': 'agendaItemDescription:af43f771-5d2d-4...,1,"{'date': '2025-07-08T18:30:00', 'entityName': ..."
186,proposal:31d554a0-4bee-49f5-a324-934c9c6820ef,proposal,2025-05-13T16:30:00.000Z,2025-05-13T18:30:00,0.299861,[{'id': 'proposalDescription:31d554a0-4bee-49f...,1,"{'date': '2025-05-13T18:30:00', 'entityName': ..."
187,proposal:f5ebb154-8d68-461e-acd3-4802334202a9,proposal,2025-05-21T12:00:00.000Z,2025-05-21T14:00:00,0.299008,[{'id': 'proposalDescription:f5ebb154-8d68-461...,1,"{'date': '2025-05-21T14:00:00', 'entityName': ..."


In [250]:
data["entity_name"] = data["context"].apply(lambda x: ast.literal_eval(x)["entityName"])
data["entity_id"] = data["context"].apply(lambda x: ast.literal_eval(x)["entityId"])
data["date"] = pd.to_datetime(data["date"], format="%Y-%m-%dT%H:%M:%S")
data = data.sort_values(by="date", ascending=False)

In [251]:
data

,groupKey,groupType,groupCreationDate,date,score,hits,totalHits,context,entity_name,entity_id
133,proposal:a49e6579-d3a6-4f23-8781-f2227c383a56,proposal,2026-05-07T10:07:37.688Z,2026-07-01 14:00:00,0.476972,[{'id': 'proposalDescription:a49e6579-d3a6-4f2...,1,"{'date': '2026-07-01T14:00:00', 'entityName': ...","Leipzig, Stadt",147130000000
177,proposal:e1c96bc7-241c-4ca8-82a1-8d7ea942524c,proposal,2026-06-24T13:15:51.476Z,2026-06-24 18:00:00,0.355593,[{'id': 'proposalDescription:e1c96bc7-241c-4ca...,1,"{'date': '2026-06-24T18:00:00', 'entityName': ...","Leipzig, Stadt",147130000000
78,meeting:18a566e5-dc2d-49f7-86d2-49f85ba5ad0f,meeting,2026-06-24T13:16:34.031Z,2026-06-24 17:30:00,0.796028,[{'id': 'document:dae2ec6c-05d2-4ea9-a11d-21be...,1,"{'date': '2026-06-24T17:30:00', 'entityName': ...","Leipzig, Stadt",147130000000
36,meeting:7bcef9ff-a8fc-4e48-8578-3d549229356b,meeting,2026-05-24T10:14:33.606Z,2026-06-02 19:00:00,1.124695,[{'id': 'agendaItemTitle:2607996a-2f9c-43b0-a6...,2,"{'date': '2026-06-02T19:00:00', 'entityName': ...","Leipzig, Stadt",147130000000
6,meeting:109fbae0-36b0-4fda-8bc8-9dd02e4faaa8,meeting,2026-06-03T18:22:03.695Z,2026-05-28 16:00:00,1.962283,[{'id': 'document:72c1127c-f941-491f-b45d-dca3...,3,"{'date': '2026-05-28T16:00:00', 'entityName': ...","Leipzig, Stadt",147130000000
...,...,...,...,...,...,...,...,...,...,...
72,meeting:e4aa30cc-94dc-4686-b749-be43a4ce7af3,meeting,2024-02-15T17:00:00.000Z,2024-02-15 18:00:00,0.867828,[{'id': 'agendaItemDescription:406a4c66-024b-4...,1,"{'date': '2024-02-15T18:00:00', 'entityName': ...","Leipzig, Stadt",147130000000
163,meeting:419fff0f-7d87-416e-81e0-fb627c77ec0d,meeting,2024-02-13T17:30:00.000Z,2024-02-13 18:30:00,0.391994,[{'id': 'agendaItemDescription:1638bf03-ab7f-4...,1,"{'date': '2024-02-13T18:30:00', 'entityName': ...","Rostock, Hanse- und Universitätsstadt",130030000
184,proposal:558ab72a-badb-4452-a05c-227e6c97220c,proposal,2024-02-07T16:00:00.000Z,2024-02-07 17:00:00,0.309111,[{'id': 'proposalDescription:558ab72a-badb-445...,1,"{'date': '2024-02-07T17:00:00', 'entityName': ...","Leipzig, Stadt",147130000000
76,meeting:26fcb33f-212a-4a63-ad58-cc874dd9f8b6,meeting,2024-01-30T17:30:00.000Z,2024-01-30 18:30:00,0.823431,[{'id': 'agendaItemDescription:c4fd5e49-859e-4...,1,"{'date': '2024-01-30T18:30:00', 'entityName': ...","Leipzig, Stadt",147130000000


In [254]:
data["entity_name"].value_counts()

entity_name
Leipzig, Stadt                           110
Rostock, Hanse- und Universitätsstadt     79
Name: count, dtype: int64

In [243]:
logs = pd.DataFrame(columns=["city_name", "entity_id", "input_tokens", "output_tokens", "costs", "mcp_usage"])
results = pd.DataFrame(columns=["city_name", "entity_id", "classification_result"])

for city in data[:1]["entity_name"].unique():
    city_data = data[data["entity_name"] == city]
    classification_base = [result["date"].strftime("%Y-%m-%d %H:%M:%S") + ":" + result["hits"] for _, result in city_data.iterrows()]
    city_data_point_count = len(city_data)
    city_name = city
    ris = city_data.iloc[0]['entity_id']
    user_prompt = f"Klassifiziere den Status der Wärmeplanung in {city_name}, mit der ID {ris}. Beziehe dich dabei auf folgende Daten und schau dir zuerst die jüngsten Ergebnisse an, da diese am wahrscheinlichsten eine Entscheidung enthalten. {classification_base}"
    response = await agent.run(user_prompt)
    
    # Tokens und MCP-Nutzung monitoren
    tokens_info = monitor_tokens(response)
    
    # In logs DataFrame speichern
    log_entry = {
        "city_name": city_name,
        "entity_id": ris,
        "input_tokens": tokens_info["input_tokens"],
        "output_tokens": tokens_info["output_tokens"],
        "costs": tokens_info["costs"],
        "mcp_usage": tokens_info["mcp_usage"]
    }
    logs = pd.concat([logs, pd.DataFrame([log_entry])], ignore_index=True)
    
    # In results DataFrame speichern
    result_entry = {
        "city_name": city_name,
        "entity_id": ris,
        "classification_result": response.output
    }
    results = pd.concat([results, pd.DataFrame([result_entry])], ignore_index=True)


In [244]:
response.all_messages()

[ModelRequest(parts=[SystemPromptPart(content='Du bist ein Experte für Klassifikation von kommunalen politischen Prozessen zur Wärmeplanung.Du liest Auszüge aus kommunalpolitischen Entscheidungs-, und Planungsprozessen.Klassifiziere diese Verfahren nach Status abgeschlossen, laufend, unklar.Falls die Auszüge nicht genügen, um sicher zu klassifizieren, kannst du den Poliscope-MCP nutzen, um weitere Dokumente auszuwerten und selbstständig durch Ratsinformationssysteme zu navigieren.Du antwortest IMMER NUR IN EINEM WORT.abgeschlossen - wenn die Wärmeplanung verabschiedet bzw beschlossen ist.laufend - wenn ein Konzept bereits erarbeitet bzw. besprochen wird, aber der Vorgang noch nicht abgeschlosen wurde.unklar - wenn du anhand der vorhandenen Informationen nicht sicher entscheiden kannst.', timestamp=datetime.datetime(2026, 6, 26, 13, 59, 56, 286031, tzinfo=datetime.timezone.utc)), UserPromptPart(content='Klassifiziere den Status der Wärmeplanung in Leipzig, Stadt, mit der ID 147130000000

In [255]:
logs

,city_name,entity_id,input_tokens,output_tokens,costs,mcp_usage
0,"Leipzig, Stadt",147130000000,83550,436,0.007598,{}


In [256]:
results


,city_name,entity_id,classification_result
0,"Leipzig, Stadt",147130000000,laufend
